In [1]:
%load_ext autoreload
%autoreload 2

import os
ROOT_PATH = os.path.join(os.getcwd(), os.pardir)
import sys
sys.path.append(ROOT_PATH) 
import pandas as pd
import numpy as np
import notebooks.utils as notebooks_utils
import warnings
warnings.filterwarnings('ignore')
from tqdm import tqdm


/tmp/ipykernel_97718/1646661068.py:8: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
#financial=pd.read_csv('/home/andreasideras/research_ideas/BankruptcyPredictionAD/data/numerical.csv')
mda_auditor=pd.read_csv('/home/andreasideras/research_ideas/BankruptcyPredictionAD/data/mda_auditor.csv')

In [3]:
mda_auditor.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39041 entries, 0 to 39040
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   uid           39041 non-null  object
 1   label         39041 non-null  bool  
 2   item_7        39041 non-null  object
 3   opinion_text  39041 non-null  object
dtypes: bool(1), object(3)
memory usage: 953.3+ KB


In [11]:
mean_density = financial.notna().sum(axis=1).mean()
std_density = financial.notna().sum(axis=1).std()
print(f"Each sample has {mean_density:.2f} ± {std_density:.2f} non-nan features")

Each sample has 210.37 ± 93.93 non-nan features


In [21]:
(financial.drop(columns=['label','uid']).notna().sum(axis=0)/financial.shape[0]).sort_values(ascending=False).head(40).apply(lambda x : f"{round(x*100,1)}%")
# Density Percentage of 20 most dense features 

CashAndCashEquivalentsAtCarryingValue                                       92.3%
IncomeTaxExpenseBenefit                                                     88.6%
StockholdersEquity                                                          86.5%
Assets                                                                      86.2%
LiabilitiesAndStockholdersEquity                                            85.6%
NetIncomeLoss                                                               84.7%
NetCashProvidedByUsedInFinancingActivities                                  81.4%
NetCashProvidedByUsedInInvestingActivities                                  81.4%
NetCashProvidedByUsedInOperatingActivities                                  81.2%
RetainedEarningsAccumulatedDeficit                                          80.3%
EarningsPerShareBasic                                                       80.2%
WeightedAverageNumberOfSharesOutstandingBasic                               79.6%
EarningsPerShare

Get the embeddings of tags' descriptions

In [ ]:
from sentence_transformers import SentenceTransformer
df = pd.read_csv("/home/andreasideras/research_ideas/BankruptcyPredictionAD/tag_description.csv")
texts = df["description"].astype(str).tolist()
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(texts, show_progress_bar=True)
df_emb = pd.DataFrame(embeddings, columns=[f"emb_{i}" for i in range(embeddings.shape[1])])
df_combined = pd.concat([df, df_emb], axis=1)
df_combined.to_csv("csv_with_embeddings.csv", index=False)


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Batches:   0%|          | 0/220 [00:00<?, ?it/s]

Bankrupt cases throught years

In [2]:
financial=pd.read_csv('/home/andreasideras/research_ideas/BankruptcyPredictionAD/data/numerical.csv')

In [ ]:
financial['year'] = financial['uid'].apply(lambda x : x.split('__')[1]).astype()

In [8]:
financial['year'] = pd.to_datetime(financial['year'])

In [15]:
financial = financial.sort_values('year').reset_index()

In [16]:
financial['year']

0       1999-12-31
1       2000-03-31
2       2000-12-31
3       2000-12-31
4       2000-12-31
           ...    
39036   2021-10-31
39037   2021-10-31
39038   2021-10-31
39039   2021-10-31
39040   2021-10-31
Name: year, Length: 39041, dtype: datetime64[ns]

In [17]:
df = financial

In [22]:
# Only positive samples
positives = df[df['label'] == 1].copy()

# Compute split indices
n_pos = len(positives)
train_end = int(0.65 * n_pos)
val_end = int(0.75 * n_pos)

# Get the date thresholds based on positive distribution
train_max_date = positives.iloc[train_end]['year']
val_max_date = positives.iloc[val_end]['year']

# Split full dataset using those thresholds
train_df = df[df['year'] <= train_max_date]
val_df = df[(df['year'] > train_max_date) & (df['year'] <= val_max_date)]
test_df = df[df['year'] > val_max_date]

# Check the number of positive samples
print('Train positives:', train_df['label'].sum())
print('Val positives:', val_df['label'].sum())
print('Test positives:', test_df['label'].sum())

Train positives: 126
Val positives: 19
Test positives: 47


In [26]:
train_df['year']

0       1999-12-31
1       2000-03-31
2       2000-12-31
3       2000-12-31
4       2000-12-31
           ...    
29747   2017-12-31
29748   2017-12-31
29749   2017-12-31
29750   2017-12-31
29751   2017-12-31
Name: year, Length: 29752, dtype: datetime64[ns]

In [27]:
val_df['year']

29752   2018-01-01
29753   2018-01-02
29754   2018-01-02
29755   2018-01-02
29756   2018-01-27
           ...    
32655   2018-12-31
32656   2018-12-31
32657   2018-12-31
32658   2018-12-31
32659   2018-12-31
Name: year, Length: 2908, dtype: datetime64[ns]

In [28]:
test_df['year']

32660   2019-01-01
32661   2019-01-01
32662   2019-01-01
32663   2019-01-26
32664   2019-01-26
           ...    
39036   2021-10-31
39037   2021-10-31
39038   2021-10-31
39039   2021-10-31
39040   2021-10-31
Name: year, Length: 6381, dtype: datetime64[ns]